# AC Lab-PC History Proof & Verification

This notebook verifies and presents the historical proof artifacts from the lab-PC training runs for the Fetal Abdominal Circumference (AC) task.

### Core Principles
- **No Hand-Typed Numbers**: Every figure, metric, and count is read directly from the committed input artifacts in `inputs/`.
- **Historic Proof Status**: These runs document iterative development on the lab PC. Because test sets were used during model exploration (flip-TTA ablations), these rows are historic proofs rather than official hold-out benchmarks.
- **Protocol Differences**: DSNT 5-fold cross-validation evaluated 789 pooled images out-of-fold, which is a fundamentally different protocol from the official test split.


In [1]:
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Ensure scripts dir is accessible
work_dir = Path.cwd().resolve()
while work_dir.name != 'work' and work_dir.parent != work_dir:
    work_dir = work_dir.parent
if str(work_dir / 'scripts') not in sys.path:
    sys.path.insert(0, str(work_dir / 'scripts'))

import importlib.util
spec = importlib.util.spec_from_file_location('ac_history_table', work_dir / 'scripts' / '28_ac_history_table.py')
ac_hist = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ac_hist)

inputs_dir = work_dir / 'notebooks' / 'history' / 'ac_lab_pc' / 'inputs'
print(f'Inputs directory: {inputs_dir}')
assert inputs_dir.is_dir()


Inputs directory: C:\fetal-github-clone\work\notebooks\history\ac_lab_pc\inputs


In [2]:
# Generate structured history data and render markdown table
data = ac_hist.generate_history_data(inputs_dir)
md_table = ac_hist.render_markdown(data)
display(Markdown(md_table))


# AC Lab-PC History Proof Table

Historical proof benchmarks extracted strictly from the committed lab-PC artifacts (`notebooks/history/ac_lab_pc/inputs/`).

## Performance Table

| Model / Run | Res | N | AC MAE (mm) | RMSE (mm) | Median (mm) | Bias (mm) | AC % | >10mm (%) | Status |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|---|
| **v1 heatmap regression (Colab)** | 384 | 789 | 182.60 | 195.13 | - | - | - | - | v1 heatmap regression, Colab, 5-fold out-of-fold (789 images): FAILED/COLLAPSED run (AC MAE 182.60 mm vs mean-AC baseline), not comparable with official-test rows |
| **DSNT 5-fold OOF** | 384 | 789 | 5.18 | 9.07 | - | - | 2.60% | - | history row, DIFFERENT protocol (789 images incl. the official-test images, out-of-fold), not comparable with the official-test rows |
| **v3 base** | 384 | 147 | 5.81 | 8.87 | 3.65 | -3.26 | 2.80% | 16.3% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_full** | 384 | 147 | 4.40 | 6.16 | 2.91 | -1.55 | 2.14% | 8.2% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (s1)** | 384 | 147 | 4.96 | 7.33 | 3.14 | +0.96 | 2.39% | 12.2% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (s2)** | 384 | 147 | 4.32 | 6.05 | 2.73 | -0.99 | 2.17% | 8.8% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (s3)** | 384 | 147 | 4.54 | 6.59 | 3.37 | -0.74 | 2.20% | 10.2% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (s4)** | 384 | 147 | 4.49 | 6.27 | 3.64 | -0.49 | 2.17% | 8.2% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (res512)** | 512 | 147 | 5.02 | 8.01 | 3.32 | -1.53 | 2.48% | 11.6% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (4-seed mean)** | 384 | 147 | 4.47 | 6.31 | 3.01 | -0.53 | 2.17% | 9.5% | ablation-study rows on the official test (test set was looked at many times) |
| **v3_ens (4-seed median)** | 384 | 147 | 4.34 | 6.03 | 3.12 | -0.60 | 2.14% | 6.8% | ablation-study rows on the official test (test set was looked at many times) |
| **smoke_log (code check)** | - | 32 | 111.65 | 117.88 | - | - | 57.72% | - | code check only, not a result |
| **v2 DSNT smoke (Colab CPU)** | 384 | 160 | - | - | - | - | - | - | DSNT smoke on Colab CPU, 1 fold, 5 epochs: code check only; the real v2 run is the lab-PC full_log row |

> **Single 384-px seeds summary**: AC MAE 4.58 ± 0.27 mm (min 4.32, max 4.96) across seeds [1, 2, 3, 4].

## Log Fact-Check Audit

Automated scan of the 4 committed UTF-16 LE execution logs:

| Log File | Lines | Invocations (`device=`) | Resumed Lines | Tracebacks | Seeds in Header | Status / Notes |
|---|:---:|:---:|:---:|:---:|---|---|
| `v3_log.txt` | 82 | 1 | 0 | 0 | - | 1 clean invocation: base + v3_full ablation. |
| `ens_log.txt` | 297 | 4 | 0 | 1 | [[1, 2, 3, 4, 5], [1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4]] | Seeds [1,2,3,4,5] requested in 1st invocation; restart switched to [1,2,3,4]; s5 was never trained. |
| `smoke_log.txt` | 49 | 1 | 0 | 0 | - | code check only, not a result |
| `full_log.txt` | 301 | 5 | 1 | 5 | - | 5 invocations (`Using device:`), 1 fold resumed. 5 crashes (RuntimeError: ios_base::badbit set: iostream stream error; RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 92174400 vs 92174288; ModuleNotFoundError: No module named 'numpy'; FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/BITC MEDIA/Documents/nor/fetal-project/data/raw/FETAL_BIOMETRY_ABDOMEN/FP/annotations/Abdomen.csv'; FileNotFoundError: [Errno 2] No such file or directory: 'PASTE_THE_PATH_HERE/FP/annotations/Abdomen.csv'), run resumed; the script refuses to resume with a different CONFIG. |

### Key Verification Assertions
1. **Seed 5 Status**: `Seed 5 was planned but never trained (no [s5] lines)`. Confirmed: exactly 0 `[s5]` lines across all logs.
2. **DSNT OOF Equality**: Computed metrics on `dsnt_full_oof_results.csv` ($N=789$, MAE=5.18 mm, RMSE=9.07 mm, % err=2.60%, FP MAE=5.08 mm, UCL MAE=5.91 mm) match `full_log.txt` final block exact to printed precision.
3. **v3 Official Test Equality**: Values in `v3_base_results.json` and `v3_full_results.json` match `v3_log.txt` test table exact to printed precision.
4. **Ensemble Consistency**: Values in `ens_results.json` match `ens_log.txt` table exact to printed precision. Recomputed MAE from `ens_test_predictions.csv` matches `ENS_mean` (4.4693 mm) exactly.

## Colab Historical Development Evidence (v1 & v2)

Preserved in committed Colab notebook (`notebooks/history/Capstone_fetalDataset_history.ipynb`):
- **v1 Heatmap Regression**: Pooled dataset: 789 images from 615 unique patients (586 FP + 29 UCL). Folds 2–3 completed 100 epochs; Fold 4 early-stopped at epoch 97 (val_loss 0.000344); Fold 5 early-stopped at epoch 89 (val_loss 0.000351). Out-of-fold metrics: TAD MAE 57.98 mm, APAD MAE 57.46 mm, AC MAE 182.60 mm (collapsed run).
- **v2 DSNT Smoke**: DSNT smoke on Colab CPU, 1 fold, 5 epochs: code check only; the real v2 run is the lab-PC full_log row.



In [3]:
# Inspect log headers and final blocks
logs = ['full_log.txt', 'v3_log.txt', 'ens_log.txt', 'smoke_log.txt']
for lname in logs:
    text = (inputs_dir / lname).read_text(encoding='utf-16')
    lines = text.splitlines()
    print(f'=== {lname} ({len(lines)} lines) ===')
    print('Header:', lines[0] if lines else '')
    for l in lines:
        if any(k in l for k in ['Using device:', 'device=', 'Resumed', 'Traceback']):
            print('  marker:', l)
    print('...')
    print('Tail:')
    for l in lines[-5:]:
        print(' ', l)
    print()


=== full_log.txt (301 lines) ===
Header: Using device: cuda | SMOKE_TEST=False | checkpoints -> C:/fetal-project/models/ac_dsnt_full
  marker: Using device: cuda | SMOKE_TEST=False | checkpoints -> C:/fetal-project/models/ac_dsnt_full
  marker: Traceback (most recent call last):
  marker: Traceback (most recent call last):
  marker: Traceback (most recent call last):
  marker: Using device: cuda | SMOKE_TEST=False | checkpoints -> C:/fetal-project/models/ac_dsnt_full
  marker: Traceback (most recent call last):
  marker: Using device: cuda | SMOKE_TEST=False | checkpoints -> C:/fetal-project/models/ac_dsnt_full
  marker: Traceback (most recent call last):
  marker: Using device: cuda | SMOKE_TEST=False | checkpoints -> C:/fetal-project/models/ac_dsnt_full
  marker:   Resumed fold 3 at epoch 1 (best val landmark error 51.12px)
  marker: Using device: cuda | SMOKE_TEST=False | checkpoints -> C:/fetal-project/models/ac_dsnt_full
...
Tail:
         (reference: always predicting the mean AC

In [4]:
# Run exact equality checks and assertions
print('--- Checking DSNT OOF Metrics ---')
df_oof = pd.read_csv(inputs_dir / 'dsnt_full_oof_results.csv')
full_log_text = (inputs_dir / 'full_log.txt').read_text(encoding='utf-16')
parsed_oof = ac_hist.parse_full_log_oof(full_log_text)
oof_err = (df_oof['ac_pred'] - df_oof['ac_true']).abs()

assert len(df_oof) == parsed_oof['n'] == 789
assert round(float(oof_err.mean()), 2) == parsed_oof['ac_mae'] == 5.18
assert round(float(np.sqrt(((df_oof['ac_pred'] - df_oof['ac_true']) ** 2).mean())), 2) == parsed_oof['ac_rmse'] == 9.07
assert round(float((100.0 * oof_err / df_oof['ac_true']).mean()), 2) == parsed_oof['mean_abs_pct_error'] == 2.60
assert round(float(oof_err[df_oof['source'] == 'FP'].mean()), 2) == parsed_oof['fp_mae'] == 5.08
assert round(float(oof_err[df_oof['source'] == 'UCL'].mean()), 2) == parsed_oof['ucl_mae'] == 5.91
print('DSNT 5-fold OOF matches full_log.txt exact to printed rounding (5.18 mm MAE).')

print('--- Checking v3 Base & Full ---')
v3_log_text = (inputs_dir / 'v3_log.txt').read_text(encoding='utf-16')
parsed_v3 = ac_hist.parse_v3_log_table(v3_log_text)
v3_b = json.loads((inputs_dir / 'v3_base_results.json').read_text())['test_tta']
v3_f = json.loads((inputs_dir / 'v3_full_results.json').read_text())['test_tta']
assert round(v3_b['ac_mae'], 2) == parsed_v3['base']['ac_mae'] == 5.81
assert round(v3_f['ac_mae'], 2) == parsed_v3['v3_full']['ac_mae'] == 4.40
print('v3_base (5.81 mm) and v3_full (4.40 mm) match v3_log.txt test table.')

print('--- Checking Seed Ensemble ---')
ens_json = json.loads((inputs_dir / 'ens_results.json').read_text())
df_ens = pd.read_csv(inputs_dir / 'ens_test_predictions.csv')
recomp_mae = float((df_ens['ac_pred'] - df_ens['ac_true']).abs().mean())
assert np.isclose(recomp_mae, ens_json['ENS_mean']['ac_mae'])
print(f'Recomputed ensemble MAE: {recomp_mae:.4f} mm matches ENS_mean exactly.')
print(f'ENS_median MAE: {ens_json["ENS_median"]["ac_mae"]:.4f} mm.')

print('--- Checking Seed 5 Status ---')
facts = ac_hist.extract_log_facts(inputs_dir)
total_s5 = sum(f['s5_lines_count'] for f in facts.values())
assert total_s5 == 0
print('Seed 5 was planned in initial header but never trained: verified 0 [s5] lines.')


--- Checking DSNT OOF Metrics ---
DSNT 5-fold OOF matches full_log.txt exact to printed rounding (5.18 mm MAE).
--- Checking v3 Base & Full ---
v3_base (5.81 mm) and v3_full (4.40 mm) match v3_log.txt test table.
--- Checking Seed Ensemble ---
Recomputed ensemble MAE: 4.4693 mm matches ENS_mean exactly.
ENS_median MAE: 4.3430 mm.
--- Checking Seed 5 Status ---
Seed 5 was planned in initial header but never trained: verified 0 [s5] lines.


In [5]:
# Compare lab_weights_sha256.txt with repo model directory
lab_weights_path = Path('C:/Users/Hp/Downloads/lab_weights_sha256.txt')
manifest_path = Path('C:/fetal-models/MANIFEST.sha256')

if lab_weights_path.is_file() and manifest_path.is_file():
    lab_entries = {}
    for line in lab_weights_path.read_text().splitlines():
        line = line.strip()
        if line:
            sha, rel = line.split(maxsplit=1)
            lab_entries[rel] = sha

    repo_manifest = {}
    for line in manifest_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#'):
            parts = line.split(maxsplit=2)
            if len(parts) == 3:
                sha, sz, rel = parts
                repo_manifest[rel] = sha

    print('=== Weights Comparison Audit ===')
    matched = []
    missing_from_repo = []
    for rel, sha in lab_entries.items():
        repo_rel = rel.removeprefix('models/')
        if repo_rel in repo_manifest:
            if repo_manifest[repo_rel] == sha:
                matched.append((rel, repo_rel, sha))
            else:
                print(f'SHA MISMATCH: {rel}')
        else:
            missing_from_repo.append((rel, sha))

    print(f'Matching lab weights in repo manifest: {len(matched)} / {len(lab_entries)}')
    for l_rel, r_rel, sha in matched:
        print(f'  [MATCH] {l_rel} -> {r_rel} ({sha[:12]}...)')

    print(f'Lab weights missing from repo manifest: {len(missing_from_repo)}')
    for rel, sha in missing_from_repo:
        print(f'  [MISSING IN REPO] {rel} ({sha[:12]}...)')
else:
    print('Manifest comparison skipped (one or both files not at expected absolute paths).')


=== Weights Comparison Audit ===
Matching lab weights in repo manifest: 6 / 7
  [MATCH] models/ac_v3/base/best.pt -> ac_v3/base/best.pt (baa348d87cd5...)
  [MATCH] models/ac_v3/v3_full/best.pt -> ac_v3/v3_full/best.pt (8e16e41ec0df...)
  [MATCH] models/ac_v3_ens/s1/best.pt -> ac_v3_ens/s1/best.pt (33fe2e6a386c...)
  [MATCH] models/ac_v3_ens/s2/best.pt -> ac_v3_ens/s2/best.pt (31b8b9c5f896...)
  [MATCH] models/ac_v3_ens/s3/best.pt -> ac_v3_ens/s3/best.pt (b69abe143b77...)
  [MATCH] models/ac_v3_ens/s4/best.pt -> ac_v3_ens/s4/best.pt (c490a8823762...)
Lab weights missing from repo manifest: 1
  [MISSING IN REPO] models/ac_v3_ens/res512/best.pt (3ed69d942660...)
